# 《一文读懂贝叶斯网络》配套教学 Notebook

本 Notebook 是[一文读懂贝叶斯网络.md](05_一文读懂贝叶斯网络.md)教案的配套动手教程。

说明：本文中的示例参数与样本均为教学构造，不对应真实业务数据集。

## 学习目标

1. 复算“感冒 → 发烧/喷嚏”网络的后验推断结果。
2. 用枚举计算在数值层面验证 d-分离的三类基本结构。
3. 从小样本中估计 CPT，并理解拉普拉斯平滑对 0 概率的修正。
4. （选学）在三变量玩具数据上做结构学习（BIC 枚举）。


## 1 感冒诊断网络：复算 $P(C=1 \mid F=1, S=1)$

来源：教案「05_一文读懂贝叶斯网络.md」第 3.4 节与第 4.4 节中的示例设定。

网络结构（文本示意）：

```text
    C
   / \
  F   S
```

参数（教学假设）：

- $P(C=1)=0.2$
- $P(F=1\mid C=1)=0.8,\;P(F=1\mid C=0)=0.1$
- $P(S=1\mid C=1)=0.7,\;P(S=1\mid C=0)=0.2$

In [28]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import product
from typing import Dict, Iterable, Tuple


@dataclass(frozen=True)
class ColdNetworkParams:
    p_c1: float
    p_f1_given_c: Dict[int, float]
    p_s1_given_c: Dict[int, float]


params = ColdNetworkParams(
    p_c1=0.2,
    p_f1_given_c={1: 0.8, 0: 0.1},
    p_s1_given_c={1: 0.7, 0: 0.2},
)


def posterior_c1_given_f1s1(p: ColdNetworkParams) -> float:
    p_c = {1: p.p_c1, 0: 1.0 - p.p_c1}
    p_joint_c1 = p_c[1] * p.p_f1_given_c[1] * p.p_s1_given_c[1]
    p_joint_c0 = p_c[0] * p.p_f1_given_c[0] * p.p_s1_given_c[0]
    return p_joint_c1 / (p_joint_c1 + p_joint_c0)


p_c = {1: params.p_c1, 0: 1.0 - params.p_c1}
p_joint_c1 = p_c[1] * params.p_f1_given_c[1] * params.p_s1_given_c[1]
p_joint_c0 = p_c[0] * params.p_f1_given_c[0] * params.p_s1_given_c[0]
posterior = posterior_c1_given_f1s1(params)

print("P(C=1,F=1,S=1) =", f"{p_joint_c1:.3f}")
print("P(C=0,F=1,S=1) =", f"{p_joint_c0:.3f}")
print("P(C=1 | F=1,S=1) =", f"{posterior:.3f}")



params_exercise = ColdNetworkParams(
    p_c1=0.05,
    p_f1_given_c=params.p_f1_given_c,
    p_s1_given_c=params.p_s1_given_c,
)
posterior_exercise = posterior_c1_given_f1s1(params_exercise)
print("\n练习：将 P(C=1) 改为 0.05 后")
print("P(C=1 | F=1,S=1) =", f"{posterior_exercise:.3f}")


P(C=1,F=1,S=1) = 0.112
P(C=0,F=1,S=1) = 0.016
P(C=1 | F=1,S=1) = 0.875

练习：将 P(C=1) 改为 0.05 后
P(C=1 | F=1,S=1) = 0.596


## 2 d-分离：用数值验证三类基本结构

目标：在二值变量的可枚举场景下，用数值等式验证以下结论：

1. 链式结构 $A \rightarrow B \rightarrow C$：$A \perp C \mid B$。
2. 分叉结构 $A \leftarrow B \rightarrow C$：$A \perp C \mid B$。
3. 碰撞结构 $A \rightarrow B \leftarrow C$：通常 $A \perp C$，但观测 $B$ 后 $A$ 与 $C$ 变得相关。

In [29]:
from itertools import product
from typing import Dict, Iterable, Tuple

def normalize(d: Dict[Tuple[int, ...], float]) -> Dict[Tuple[int, ...], float]:
    s = sum(d.values())
    if s <= 0:
        raise ValueError("total probability mass must be positive")
    return {k: v / s for k, v in d.items()}


def marginalize(
    p: Dict[Tuple[int, ...], float],
    var_indices: Iterable[int],
) -> Dict[Tuple[int, ...], float]:
    idx = tuple(var_indices)
    out: Dict[Tuple[int, ...], float] = {}
    for key, value in p.items():
        subkey = tuple(key[i] for i in idx)
        out[subkey] = out.get(subkey, 0.0) + value
    return out


def conditional(
    p_xyz: Dict[Tuple[int, int, int], float],
    given_index: int,
    given_value: int,
) -> Dict[Tuple[int, int, int], float]:
    filtered = {k: v for k, v in p_xyz.items() if k[given_index] == given_value}
    return normalize(filtered)


def max_abs_independence_gap_marginal(
    p_xyz: Dict[Tuple[int, int, int], float],
    x_index: int,
    y_index: int,
) -> float:
    px = marginalize(p_xyz, [x_index])
    py = marginalize(p_xyz, [y_index])
    pxy = marginalize(p_xyz, [x_index, y_index])

    gap = 0.0
    for x in [0, 1]:
        for y in [0, 1]:
            pxy_val = pxy[(x, y)]
            prod = px[(x,)] * py[(y,)]
            gap = max(gap, abs(pxy_val - prod))
    return gap


def max_abs_independence_gap_conditional(
    p_xyz: Dict[Tuple[int, int, int], float],
    x_index: int,
    y_index: int,
    z_index: int,
) -> float:
    gap = 0.0
    for z in [0, 1]:
        p_cond = conditional(p_xyz, given_index=z_index, given_value=z)
        px = marginalize(p_cond, [x_index])
        py = marginalize(p_cond, [y_index])
        pxy = marginalize(p_cond, [x_index, y_index])
        for x in [0, 1]:
            for y in [0, 1]:
                pxy_val = pxy[(x, y)]
                prod = px[(x,)] * py[(y,)]
                gap = max(gap, abs(pxy_val - prod))
    return gap


def build_chain_joint() -> Dict[Tuple[int, int, int], float]:
    p_a1 = 0.3
    p_b1_given_a = {1: 0.8, 0: 0.2}
    p_c1_given_b = {1: 0.9, 0: 0.1}

    out: Dict[Tuple[int, int, int], float] = {}
    for a, b, c in product([0, 1], repeat=3):
        p_a = p_a1 if a == 1 else 1.0 - p_a1
        p_b = p_b1_given_a[a] if b == 1 else 1.0 - p_b1_given_a[a]
        p_c = p_c1_given_b[b] if c == 1 else 1.0 - p_c1_given_b[b]
        out[(a, b, c)] = p_a * p_b * p_c
    return out


def build_fork_joint() -> Dict[Tuple[int, int, int], float]:
    p_b1 = 0.4
    p_a1_given_b = {1: 0.7, 0: 0.2}
    p_c1_given_b = {1: 0.6, 0: 0.1}

    out: Dict[Tuple[int, int, int], float] = {}
    for a, b, c in product([0, 1], repeat=3):
        p_b = p_b1 if b == 1 else 1.0 - p_b1
        p_a = p_a1_given_b[b] if a == 1 else 1.0 - p_a1_given_b[b]
        p_c = p_c1_given_b[b] if c == 1 else 1.0 - p_c1_given_b[b]
        out[(a, b, c)] = p_b * p_a * p_c
    return out


def build_collider_joint() -> Dict[Tuple[int, int, int], float]:
    p_a1 = 0.3
    p_c1 = 0.5
    p_b1_given_ac = {
        (0, 0): 0.05,
        (0, 1): 0.6,
        (1, 0): 0.7,
        (1, 1): 0.95,
    }

    out: Dict[Tuple[int, int, int], float] = {}
    for a, b, c in product([0, 1], repeat=3):
        p_a = p_a1 if a == 1 else 1.0 - p_a1
        p_c = p_c1 if c == 1 else 1.0 - p_c1
        p_b1 = p_b1_given_ac[(a, c)]
        p_b = p_b1 if b == 1 else 1.0 - p_b1
        out[(a, b, c)] = p_a * p_c * p_b
    return out


p_chain = build_chain_joint()
gap_chain_marginal = max_abs_independence_gap_marginal(p_chain, x_index=0, y_index=2)
gap_chain_cond = max_abs_independence_gap_conditional(p_chain, x_index=0, y_index=2, z_index=1)
print("链式结构 A->B->C")
print("max |P(A,C) - P(A)P(C)| =", f"{gap_chain_marginal:.6f}")
print("max |P(A,C|B) - P(A|B)P(C|B)| =", f"{gap_chain_cond:.6f}")


p_fork = build_fork_joint()
gap_fork_marginal = max_abs_independence_gap_marginal(p_fork, x_index=0, y_index=2)
gap_fork_cond = max_abs_independence_gap_conditional(p_fork, x_index=0, y_index=2, z_index=1)
print("\n分叉结构 A<-B->C")
print("max |P(A,C) - P(A)P(C)| =", f"{gap_fork_marginal:.6f}")
print("max |P(A,C|B) - P(A|B)P(C|B)| =", f"{gap_fork_cond:.6f}")


p_collider = build_collider_joint()
gap_collider_marginal = max_abs_independence_gap_marginal(p_collider, x_index=0, y_index=2)
gap_collider_cond_on_b1 = max_abs_independence_gap_conditional(p_collider, x_index=0, y_index=2, z_index=1)
print("\n碰撞结构 A->B<-C")
print("max |P(A,C) - P(A)P(C)| =", f"{gap_collider_marginal:.6f}")
print("max |P(A,C|B) - P(A|B)P(C|B)| =", f"{gap_collider_cond_on_b1:.6f}")


链式结构 A->B->C
max |P(A,C) - P(A)P(C)| = 0.100800
max |P(A,C|B) - P(A|B)P(C|B)| = 0.000000

分叉结构 A<-B->C
max |P(A,C) - P(A)P(C)| = 0.060000
max |P(A,C|B) - P(A|B)P(C|B)| = 0.000000

碰撞结构 A->B<-C
max |P(A,C) - P(A)P(C)| = 0.000000
max |P(A,C|B) - P(A|B)P(C|B)| = 0.086676


## 3 MAP 推断：从后验到最可能解释

来源：教案「05_一文读懂贝叶斯网络.md」第 4.6 节。

在本示例里，待推断变量只有 $C$，因此 MAP 的含义是：

$$
C^* = \arg\max_{c \in \{0,1\}} P(C=c \mid F=1,S=1)
$$

由于 $P(C \mid F,S) \propto P(C)P(F\mid C)P(S\mid C)$，可以用数值直接比较两种取值对应的未归一化后验权重。

In [30]:
unnormalized = {
    1: p_c[1] * params.p_f1_given_c[1] * params.p_s1_given_c[1],
    0: p_c[0] * params.p_f1_given_c[0] * params.p_s1_given_c[0],
}
c_map = 1 if unnormalized[1] >= unnormalized[0] else 0

print("未归一化权重：", {k: round(v, 6) for k, v in unnormalized.items()})
print("MAP 结果：C*=", c_map)


未归一化权重： {1: 0.112, 0: 0.016}
MAP 结果：C*= 1


## 4 参数学习：从 5 条样本估计 CPT（含拉普拉斯平滑）

来源：教案「05_一文读懂贝叶斯网络.md」第 6.3 节的 5 条观测样本（教学构造）。

样本（C, F, S 均为二值，取值为 0/1）：

| 样本 | C | F | S |
| ---- | - | - | - |
| 1    | 1 | 1 | 1 |
| 2    | 1 | 1 | 0 |
| 3    | 1 | 0 | 1 |
| 4    | 0 | 1 | 0 |
| 5    | 0 | 0 | 0 |

In [31]:
samples = [
    {"C": 1, "F": 1, "S": 1},
    {"C": 1, "F": 1, "S": 0},
    {"C": 1, "F": 0, "S": 1},
    {"C": 0, "F": 1, "S": 0},
    {"C": 0, "F": 0, "S": 0},
]


def count_var(var: str, value: int) -> int:
    return sum(1 for row in samples if row[var] == value)


def count_pair(var_x: str, x: int, var_y: str, y: int) -> int:
    return sum(1 for row in samples if row[var_x] == x and row[var_y] == y)


n = len(samples)

p_c1_mle = count_var("C", 1) / n
p_c0_mle = 1.0 - p_c1_mle

p_f1_given_c_mle = {
    1: count_pair("F", 1, "C", 1) / count_var("C", 1),
    0: count_pair("F", 1, "C", 0) / count_var("C", 0),
}

p_s1_given_c_mle = {
    1: count_pair("S", 1, "C", 1) / count_var("C", 1),
    0: count_pair("S", 1, "C", 0) / count_var("C", 0),
}

print("MLE 估计：")
print("P(C=1)=", f"{p_c1_mle:.3f}", "P(C=0)=", f"{p_c0_mle:.3f}")
print("P(F=1|C=1)=", f"{p_f1_given_c_mle[1]:.3f}", "P(F=1|C=0)=", f"{p_f1_given_c_mle[0]:.3f}")
print("P(S=1|C=1)=", f"{p_s1_given_c_mle[1]:.3f}", "P(S=1|C=0)=", f"{p_s1_given_c_mle[0]:.3f}")


def laplace_smooth_binary(count_1: int, total: int, alpha: float = 1.0) -> float:
    k = 2
    return (count_1 + alpha) / (total + alpha * k)


alpha = 1.0
p_s1_given_c_smoothed = {
    1: laplace_smooth_binary(count_pair("S", 1, "C", 1), count_var("C", 1), alpha=alpha),
    0: laplace_smooth_binary(count_pair("S", 1, "C", 0), count_var("C", 0), alpha=alpha),
}

print("\n拉普拉斯平滑（alpha=1）后：")
print("P(S=1|C=1)=", f"{p_s1_given_c_smoothed[1]:.3f}", "P(S=1|C=0)=", f"{p_s1_given_c_smoothed[0]:.3f}")


MLE 估计：
P(C=1)= 0.600 P(C=0)= 0.400
P(F=1|C=1)= 0.667 P(F=1|C=0)= 0.500
P(S=1|C=1)= 0.667 P(S=1|C=0)= 0.000

拉普拉斯平滑（alpha=1）后：
P(S=1|C=1)= 0.600 P(S=1|C=0)= 0.250


## 5 结构学习（选学）：三变量场景的 DAG 枚举 + BIC 评分

来源：教案「05_一文读懂贝叶斯网络.md」第 5 章“结构学习”的方法概览与挑战讨论。

说明：这里用同一份 5 条样本在 $\{C,F,S\}$ 三变量上做“极小规模”的结构学习演示：

- 穷举 3 个节点的所有有向无环图（DAG）；
- 对每个结构用二值离散变量的 MLE 计算对数似然；
- 用 BIC 对模型复杂度做惩罚，选择分数最高的结构。

由于样本极少，本节输出仅用于理解“评分 + 搜索”的思想，不应解读为稳定的结构发现结论。


In [32]:
import math
from itertools import combinations

nodes = ["C", "F", "S"]
n = len(samples)

def is_acyclic(nodes, edges):
    out_edges = {u: [] for u in nodes}
    indeg = {u: 0 for u in nodes}
    for u, v in edges:
        out_edges[u].append(v)
        indeg[v] += 1
    queue = [u for u in nodes if indeg[u] == 0]
    seen = 0
    while queue:
        u = queue.pop()
        seen += 1
        for v in out_edges[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    return seen == len(nodes)

def parents_of(nodes, edges):
    pa = {u: [] for u in nodes}
    for u, v in edges:
        pa[v].append(u)
    for u in nodes:
        pa[u].sort()
    return pa

def term(count, prob):
    if count == 0:
        return 0.0
    if prob <= 0.0:
        return float("-inf")
    if prob >= 1.0:
        return float("-inf")
    return count * math.log(prob)

def local_loglik(child, parents):
    stats = {}
    for row in samples:
        key = tuple(row[p] for p in parents)
        c = row[child]
        c1, total = stats.get(key, (0, 0))
        stats[key] = (c1 + (1 if c == 1 else 0), total + 1)

    ll = 0.0
    for key, (c1, total) in stats.items():
        c0 = total - c1
        p1 = c1 / total
        p0 = 1.0 - p1
        ll += term(c1, p1)
        ll += term(c0, p0)
    return ll

def num_free_params(parents):
    return 2 ** len(parents)

all_pairs = [(u, v) for u in nodes for v in nodes if u != v]
dags = []
for k in range(len(all_pairs) + 1):
    for edges in combinations(all_pairs, k):
        if is_acyclic(nodes, edges):
            dags.append(tuple(sorted(edges)))

scored = []
for edges in dags:
    pa = parents_of(nodes, edges)
    ll = 0.0
    k_params = 0
    for x in nodes:
        ll += local_loglik(x, pa[x])
        k_params += num_free_params(pa[x])
    bic = ll - 0.5 * k_params * math.log(n)
    scored.append((bic, ll, k_params, edges))

scored.sort(reverse=True, key=lambda t: t[0])
best_bic, best_ll, best_k, best_edges = scored[0]

print("Top-5 结构（按 BIC 降序）：")
for bic, ll, k_params, edges in scored[:5]:
    print({"edges": list(edges), "loglik": round(ll, 6), "k": k_params, "bic": round(bic, 6)})

print("\nBIC 最优结构：", list(best_edges))


Top-5 结构（按 BIC 降序）：
{'edges': [], 'loglik': -10.095175, 'k': 3, 'bic': -12.509332}
{'edges': [('C', 'F')], 'loglik': -10.025954, 'k': 4, 'bic': -13.244829}
{'edges': [('F', 'C')], 'loglik': -10.025954, 'k': 4, 'bic': -13.244829}
{'edges': [('F', 'S')], 'loglik': -10.025954, 'k': 4, 'bic': -13.244829}
{'edges': [('S', 'F')], 'loglik': -10.025954, 'k': 4, 'bic': -13.244829}

BIC 最优结构： []


## 6 可选：使用 pgmpy 做推理与参数学习

本节依赖 `pgmpy` 与 `pandas`。注意：在部分环境中，`pgmpy` 可能与当前的 `scipy` 版本不兼容，导致导入失败；此时可先跳过本节，不影响前 1–5 节的复算与推导。

1. 用固定 CPT 构建模型并做精确推理；
2. 对小样本做最大似然参数学习并打印 CPT。

In [33]:
!pip install -U -i https://mirrors.cloud.tencent.com/pypi/simple pgmpy

Looking in indexes: https://mirrors.cloud.tencent.com/pypi/simple


In [34]:
try:
    import pandas as pd
    from pgmpy.factors.discrete import TabularCPD
    from pgmpy.inference import VariableElimination

    try:
        from pgmpy.models import BayesianNetwork
    except Exception:
        from pgmpy.models import BayesianModel as BayesianNetwork

    from pgmpy.estimators import MaximumLikelihoodEstimator
except Exception as e:
    msg = repr(e)
    if "_lazywhere" in msg:
        print("pgmpy 导入失败：与当前 scipy 版本不兼容（_lazywhere 相关）。建议：先跳过本节，或在独立环境中调整 scipy/pgmpy 版本后再运行。")
    else:
        print("未检测到 pgmpy 或 pandas，跳过本节。原因：", msg)
else:
    model = BayesianNetwork([("C", "F"), ("C", "S")])

    cpd_c = TabularCPD(variable="C", variable_card=2, values=[[0.8], [0.2]])
    cpd_f = TabularCPD(
        variable="F",
        variable_card=2,
        values=[
            [0.9, 0.2],
            [0.1, 0.8],
        ],
        evidence=["C"],
        evidence_card=[2],
    )
    cpd_s = TabularCPD(
        variable="S",
        variable_card=2,
        values=[
            [0.8, 0.3],
            [0.2, 0.7],
        ],
        evidence=["C"],
        evidence_card=[2],
    )

    model.add_cpds(cpd_c, cpd_f, cpd_s)

    infer = VariableElimination(model)
    q = infer.query(variables=["C"], evidence={"F": 1, "S": 1})
    print("\n推理：P(C | F=1,S=1)")
    print(q)

    df = pd.DataFrame(samples)
    model_fit = BayesianNetwork([("C", "F"), ("C", "S")])
    model_fit.fit(df, estimator=MaximumLikelihoodEstimator)
    print("\n参数学习（MLE）得到的 CPT：")
    for cpd in model_fit.get_cpds():
        print(cpd)


pgmpy 导入失败：与当前 scipy 版本不兼容（_lazywhere 相关）。建议：先跳过本节，或在独立环境中调整 scipy/pgmpy 版本后再运行。


### 安装提示（可选）

若你确实需要运行第 5 节的 `pgmpy` 示例，建议在独立环境中安装，并优先使用与你当前 `numpy/scipy` 组合兼容的版本组合。

```bash
# 可选：安装 pgmpy（可能触发额外依赖下载，耗时较长）
pip install -i https://mirrors.cloud.tencent.com/pypi/simple pgmpy
```
